# 03 — Kelime Ağırlıkları (LIME)

BERT bir yorumu sınıflandırırken **hangi kelimelerin** kararı etkilediğini LIME ile gösteriyoruz.

- **Yeşil** = pozitif yönde katkı (model 'pozitif' kararını destekliyor)
- **Kırmızı** = negatif yönde katkı (model 'negatif' kararını destekliyor)

LIME, modeli yerel olarak doğrusal bir vekil ile yaklaşık tahmin eder: bir yorumdaki kelimeleri tek tek silip modelin olasılık çıktısının nasıl değiştiğine bakar.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer
from IPython.display import HTML, display

from src.models.bert import SENTIMENT_CLASSES, load_model, predict_3class

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
VISUALS_DIR.mkdir(parents=True, exist_ok=True)
EXPL_DIR = VISUALS_DIR / 'lime'
EXPL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
_ = load_model()
explainer = LimeTextExplainer(class_names=SENTIMENT_CLASSES, bow=False)

def predict_fn(texts):
    return predict_3class(list(texts), batch_size=16)

print('Hazır.')

In [ ]:
app_store = pd.read_csv(RAW_DIR / 'app_store_reviews.csv')
google_play = pd.read_csv(RAW_DIR / 'google_play_reviews.csv')
df = pd.concat([app_store, google_play], ignore_index=True)
df['text'] = df['text'].fillna('').astype(str)
df['title'] = df['title'].fillna('').astype(str)
df['full_text'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['full_text'].str.split().str.len().between(8, 40)].copy()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('Int64')
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)
print(f'8-40 kelime arası yorum: {len(df):,}')

## Birkaç örnek yorum için açıklama

Her sınıftan 2'şer örnek seçip LIME ile inceliyoruz. Açıklamalar `visuals/lime/` altına HTML olarak kaydedilir, notebook'ta da görünür.

In [ ]:
samples = pd.concat([
    df[df['rating'] <= 2].sample(2, random_state=7),
    df[df['rating'] == 3].sample(2, random_state=7),
    df[df['rating'] >= 4].sample(2, random_state=7),
]).reset_index(drop=True)
samples[['app_name', 'rating', 'full_text']]

In [ ]:
NUM_FEATURES = 10
NUM_SAMPLES = 200

for i, row in samples.iterrows():
    text = row['full_text']
    print(f'--- Örnek {i+1} — gerçek yıldız: {row["rating"]} ({row["app_name"]}) ---')
    print(text)
    probs = predict_fn([text])[0]
    pred_idx = int(np.argmax(probs))
    print(f'Tahmin: {SENTIMENT_CLASSES[pred_idx]} (olasılıklar: '
          f'neg={probs[0]:.2f}, nötr={probs[1]:.2f}, poz={probs[2]:.2f})')

    exp = explainer.explain_instance(
        text,
        predict_fn,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES,
        labels=[0, 2],
    )
    print('  → en etkili kelimeler (pozitif sınıf için):')
    for word, weight in exp.as_list(label=2):
        print(f'    {weight:+.3f}  {word}')

    html_path = EXPL_DIR / f'example_{i+1}_rating{row["rating"]}.html'
    exp.save_to_file(str(html_path))
    print(f'  HTML kaydedildi: {html_path}')
    display(HTML(exp.as_html(labels=[2])))
    print()

## Sınıf bazlı toplu kelime ağırlıkları (opsiyonel — yavaş)

Birçok örnek üzerinde LIME çalıştırıp kelime ağırlıklarını ortalamak istersek. CPU'da yavaştır; küçük tutuyoruz.

In [ ]:
from collections import defaultdict

AGG_PER_CLASS = 10  # her sınıftan kaç yorum

agg_samples = pd.concat([
    df[df['rating'] <= 2].sample(AGG_PER_CLASS, random_state=1),
    df[df['rating'] >= 4].sample(AGG_PER_CLASS, random_state=1),
])

word_weights = defaultdict(list)
for text in agg_samples['full_text']:
    exp = explainer.explain_instance(
        text, predict_fn,
        num_features=15, num_samples=150, labels=[2],
    )
    for word, w in exp.as_list(label=2):
        word_weights[word.lower()].append(w)

agg = pd.DataFrame([
    {'kelime': w, 'ortalama_ağırlık': np.mean(ws), 'gözlem': len(ws)}
    for w, ws in word_weights.items() if len(ws) >= 2
]).sort_values('ortalama_ağırlık')

print('En negatif sinyaller (model tarafından):')
display(agg.head(15))
print('En pozitif sinyaller (model tarafından):')
display(agg.tail(15))

agg.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'lime_aggregate_weights.csv', index=False)

## Çıktılar

- `visuals/lime/example_{i}_rating{r}.html` — örnek başına LIME görselleştirmesi
- `data/processed/lime_aggregate_weights.csv` — birkaç örnek üzerinde ortalama kelime ağırlıkları

**Sonraki adım:** `streamlit run src/app/streamlit_app.py` — kullanıcı arayüzü (yorum analiz + uygulama arama).